# Glossary Translation using a Small Language Model (SLM)

**Task:** Extract an English glossary from a PDF and translate it into Tamil using an open-source Small Language Model (SLM).

**Model used:** `facebook/nllb-200-distilled-600M` (Meta's No Language Left Behind model, distilled to 600M parameters)

**Domain:** Agriculture

**Pipeline:**
1. Extract text (terms + definitions) from the glossary PDF
2. Load the NLLB SLM and its tokenizer
3. Translate each term and definition from English → Tamil
4. Display and save the translated glossary

## Step 1: Install dependencies

In [1]:
!pip install -q transformers torch sentencepiece pdfplumber accelerate


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Extract text from the glossary PDF
Since the glossary PDF is formatted as `Term: Definition` lines, we split each line on the first colon to separate the term from its definition.

In [3]:
import pdfplumber

PDF_PATH = "../data/glossary_agriculture.pdf"  # update this path if needed

def extract_glossary(pdf_path):
    """Extracts (term, definition) pairs from the glossary PDF."""
    entries = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            for line in text.split("\n"):
                line = line.strip()
                if ":" in line:
                    term, definition = line.split(":", 1)
                    entries.append((term.strip(), definition.strip()))
    return entries

glossary_entries = extract_glossary(PDF_PATH)
print(f"Extracted {len(glossary_entries)} glossary entries.\n")
for term, definition in glossary_entries[:5]:
    print(f"{term}: {definition}")

Extracted 104 glossary entries.

Activity status: see economic activity status.
Agricultural census: collection of structural data from agricultural holdings.
Agricultural holder: person making the major decisions on the operations of the holding (paragraph
Agricultural holding: economic unit of agricultural production – the basic unit of enumeration in the
Agricultural household: a household whose largest source of income consists of income derived from


## Step 3: Load the SLM (NLLB-200 distilled)
This downloads the model from Hugging Face the first time you run it (~2.4GB) and caches it locally for future runs. No API key is needed since this is an open-source model.

Note: the model file itself is not included in this GitHub repository (it would be far too large to commit). It was downloaded once during development by running this exact cell, and is now cached locally. Anyone cloning this repo will have the model auto-download the first time they run this cell — no manual download or setup required.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/nllb-200-distilled-600M"

print("Loading tokenizer and model... this may take a minute on first run.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded successfully on {device}.")

c:\Users\srini\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer and model... this may take a minute on first run.


c:\Users\srini\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\srini\.cache\huggingface\hub\models--facebook--nllb-200-distilled-600M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 697.08it/s, Materializing

Model loaded successfully on cpu.


## Step 4: Define the translation function
NLLB requires language codes in the FLORES-200 format:
- English = `eng_Latn`
- Tamil = `tam_Taml`

In [5]:
SRC_LANG = "eng_Latn"
TGT_LANG = "tam_Taml"  # change to "mal_Mlym" for Malayalam

tokenizer.src_lang = SRC_LANG

def translate_text(text, max_length=200):
    """Translates a single string from English to the target language."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# quick sanity check
print(translate_text("Irrigation is the artificial application of water to land."))

பாசனம் என்பது நிலத்தில் தண்ணீரை செயற்கையாகப் பயன்படுத்துவது.


## Step 5: Translate the full glossary
We loop through every extracted (term, definition) pair and translate both parts.

In [6]:
translated_glossary = []

for term, definition in glossary_entries:
    translated_term = translate_text(term)
    translated_definition = translate_text(definition)
    translated_glossary.append((term, definition, translated_term, translated_definition))
    print(f"EN: {term} -> TA: {translated_term}")
    print(f"    {definition}")
    print(f"    {translated_definition}\n")

EN: Activity status -> TA: செயல்பாட்டு நிலைமை
    see economic activity status.
    பொருளாதார செயல்பாட்டு நிலையை பார்க்கவும்.

EN: Agricultural census -> TA: விவசாய மக்கள் தொகை கணக்கெடுப்பு
    collection of structural data from agricultural holdings.
    விவசாய பண்ணைகளிலிருந்து கட்டமைப்பு தரவுகளை சேகரித்தல்.

EN: Agricultural holder -> TA: விவசாய வைத்திருப்பவர்
    person making the major decisions on the operations of the holding (paragraph
    ஹோல்டிங் செயல்பாடுகளில் முக்கிய முடிவுகளை எடுக்கும் நபர் (பகுதி)

EN: Agricultural holding -> TA: விவசாய ஹோல்டிங்
    economic unit of agricultural production – the basic unit of enumeration in the
    விவசாய உற்பத்தியின் பொருளாதார அலகு  வேளாண் உற்பத்தியின் அடிப்படை அலகு 

EN: Agricultural household -> TA: விவசாய குடும்பம்
    a household whose largest source of income consists of income derived from
    ஒரு குடும்பம், அதன் வருமானத்தின் மிகப்பெரிய ஆதாரம், வருமானம் பெறப்பட்ட வருமானம் ஆகும்.

EN: Agricultural land -> TA: விவசாய நிலம்
    the tot

## Step 6: Save the output


In [8]:
import os

os.makedirs("../output", exist_ok=True)
output_path = "../output/translated_glossary.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for term, definition, t_term, t_def in translated_glossary:
        f.write(f"{term} | {definition}\n")
        f.write(f"{t_term} | {t_def}\n\n")

print(f"Saved translated glossary to {output_path}")

Saved translated glossary to ../output/translated_glossary.txt
